# project_20_co2_metalloenzyme — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — metalloenzyme design, CO₂ hydration, and the Zn-His₃-OH site

**Standard slot:** *define & explore.* **For Project 20 this means:** understand de novo
**metalloenzyme** design (the GRACE paradigm) and the carbonic-anhydrase CO₂-hydration reaction, then
**construct the metal-site theozyme** (a **Zn-His₃-OH** centre + transition-state geometry) and run a
mock theozyme→scaffold hello-world (D0).

Run `00_setup.ipynb` first in this session.

## Why carbonic anhydrase is the model CO₂-hydration metalloenzyme
Carbon capture needs fast, robust CO₂-hydration catalysts (CO₂ + H₂O ⇌ HCO₃⁻ + H⁺). **Carbonic
anhydrase (CA)** is nature's champion — a small **Zn-metalloenzyme** running near the diffusion limit.
Two properties make it the model for de novo **metal-aware** design:
- **A single, well-defined catalytic metal** — a tetrahedral **Zn(II)** held by three histidines,
  with a Zn-bound hydroxide as the nucleophile. A clean, reproducible target geometry.
- **Tractable readouts** — the classic **Wilbur-Anderson** CO₂-hydration assay, plus a promiscuous
  **esterase** activity on **p-nitrophenyl acetate (pNPA)** that gives a fast chromogenic proxy.

The honest history: **GRACE** (Hu 2024) produced functional carbonic-anhydrase-style designs — but
only by generating a **large pool (~10k)** and screening. That is the methods claim this project
tests, and the expectation it sets: *diversity before filtering, then a real assay.*

## The metal-site theozyme — the catalytic motif you must build
A **theozyme** ("theoretical enzyme") is the minimal catalytic motif placed around the **transition
state**. For a metalloenzyme that motif is the **metal centre**. For carbonic anhydrase:

| Role | Residue / species | Job in the reaction |
|------|-------------------|---------------------|
| metal ion | Zn(II) | the catalytic centre; lowers the bound-water pKa to ~7 |
| 3 × His ligands | His (imidazole N) | hold the Zn in a tetrahedral cage (**fix these in design**) |
| Zn-hydroxide | OH⁻ on Zn | the nucleophile that attacks the CO₂ carbon |
| proton shuttle | His (often a 4th) | relays the proton to bulk solvent |

You **construct** this from a verified CA structure and/or a QM transition-state model — it is a
teaching template (`data/inputs/metal_site_def.txt`), **not** fabricated experimental data. Place
groups around the **transition state**, not the resting state. This is the **enzyme-family template**
(Project 18) specialised to a **metal active site**.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the metal-site theozyme (mock hello-world)
`scripts/enzyme_tools.py` exposes `build_theozyme("co2_hydration")` → a Zn-His₃-OH metal-site spec.
The distances/angles it ships are **PLACEHOLDERS** — replace them in `data/inputs/metal_site_def.txt`
(and in `build_theozyme`) with real Zn-N distances / N-Zn-N angles read off a verified CA structure
(2CAB / 3KS3 — verify on RCSB) during P1.

In [ ]:
from enzyme_tools import build_theozyme

theo = build_theozyme("co2_hydration")
print("Reaction :", theo.reaction)
print("Substrate:", theo.substrate)
print("Provenance:", theo.provenance)
site = theo.metal_site
print(f"\nMetal centre: {site.metal} | coordination {site.coordination_number} "
      f"(tetrahedral) | {site.n_protein_ligands} His ligands | reactive species: {site.reactive_species}")
print("\nCatalytic functional groups (PLACEHOLDER geometry — fill from a CA structure / QM):")
for fg in theo.functional_groups:
    ang = f"{fg.target_angle}deg" if fg.target_angle is not None else "n/a"
    print(f"  {fg.role:16s} {fg.residue}/{fg.atom:4s}  d={fg.target_distance}A  angle={ang}")
print("\nMetal ligands to FIX during sequence design:", theo.catalytic_residue_ids())

## A first mock scaffold + metal-aware sequence (no GPU)
`scaffold_motif(...)` (mock) returns placeholder backbones presenting the Zn-His₃ motif;
`ligandmpnn_metal(...)` (mock) designs sequences with the **three His ligands fixed and the Zn in
context** — the central, metal-aware step. **Every number here is SYNTHETIC** — this only proves the
plumbing runs anywhere. Switch to the real backends (RFdiffusion2/Riff-Diff on an A100; metal-aware
LigandMPNN CPU-fast) in `02_generate.ipynb`.

In [ ]:
from enzyme_tools import scaffold_motif, ligandmpnn_metal, metal_ligand_geometry

scaffolds = scaffold_motif(theo, n=5, method="mock")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_metal(scaffolds[0], theo.catalytic_residue_ids(), n=3, tool="mock")
print(f"\n{len(seqs)} mock METAL-AWARE sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_metal_ligand_roles']}, metal_aware={seqs[0]['metal_aware']})")
print("  fixed His positions (mock, 0-indexed):", seqs[0]['fixed_his_positions'])

geo = metal_ligand_geometry(None, theo.metal_site)   # mock, SYNTHETIC
print(f"\nmetal_ligand_geometry (mock, SYNTHETIC): rmsd={geo['metal_ligand_rmsd']}A  "
      f"Zn-N={geo['mean_zn_n_dist']}A  N-Zn-N={geo['mean_n_zn_n_angle']}deg  "
      f"coordination_ok={geo['coordination_ok']}  -> pass if rmsd < 0.5 A")
print("NOTE: these are placeholder numbers. The real campaign is in notebook 02.")

## The metrics that decide a metalloenzyme design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | activity |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / catalysis |
| pLDDT (catalytic) | ≥ 90 | confidence *at the His ligands* | the cage geometry is correct |
| **metal-ligand RMSD** (= `catalytic_geom_rmsd`) | **< 0.5 Å** | predicted Zn-coordinating atoms vs the target Zn-His₃ | **activity, or that the metal even binds** |

The fourth row is the point of the whole project. And the last column carries **two** messages a
metalloenzyme designer must never forget: in-silico metal geometry does not guarantee a working
enzyme, **and** good geometry does not even guarantee the **metal binds** — that is a separate,
measured check (**ICP**, notebook 05). Only an assay decides activity; only ICP decides incorporation.

> **AF2 caveat:** AF2 does **not** place the Zn. You build the metal in from the His₃ geometry (or use
> a metal-aware predictor) before scoring the metal-ligand geometry — see notebook 02/04.

## D0 checklist
- [ ] Half-page on de novo metalloenzyme design + the honest GRACE hit-rate story (large pool + screening).
- [ ] 1-page problem statement with **measurable** success criteria + the controls you'll need (apo, natural CA).
- [ ] Metal-site spec started in `data/inputs/metal_site_def.txt` (replace PLACEHOLDERs with real Zn-N distances; cite the CA structure).
- [ ] Reproduced mock hello-world (Zn-His₃-OH spec + a mock scaffold record).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the metal motif and run **metal-aware LigandMPNN** with the three His ligands fixed.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — theozyme → scaffold → metal-aware LigandMPNN (His₃ fixed, Zn context)

**Standard slot:** *design campaign.* **For Project 20 this means:** take the Zn-His₃-OH metal-site
theozyme, scaffold it into a **large pool** of backbones (RFdiffusion2 / Riff-Diff — the **A100**
step; GRACE used ~10k), then **metal-aware LigandMPNN sequence design fixing the three His ligands and
passing the Zn as context** (plus a metal-blind ProteinMPNN baseline for the benchmark), and write a
results CSV (D2).

Runs end-to-end on the **mock** backend with no GPU; switch to the real backends on Colab/HPC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams
Tools change. Before a campaign, HTTP-check that the pinned upstream repos still exist, and pin the
commit/tag you actually use. **RFdiffusion2, Riff-Diff, and CLEAN are new / fast-moving — VERIFY the
current public release/repo at generation time** (do not assert a repo you are unsure of); the others
below are stable enough to head-check.

In [ ]:
import requests

# Pinned upstreams (pin the COMMIT/TAG you use in env/requirements.txt + LOG.md):
STABLE_UPSTREAMS = {
    "RFdiffusion (classic motif scaffolding)": "https://github.com/RosettaCommons/RFdiffusion",
    "LigandMPNN (metal-aware seq design — CENTRAL)": "https://github.com/dauparas/LigandMPNN",
    "ProteinMPNN (metal-blind baseline)": "https://github.com/dauparas/ProteinMPNN",
    "AutoDock Vina (substrate fit)": "https://github.com/ccsb-scripps/AutoDock-Vina",
    "OpenMM (metal-site MD)": "https://github.com/openmm/openmm",
}
# VERIFY-ONLY (new/fast-moving; confirm the current release before relying on a URL):
VERIFY_UPSTREAMS = [
    "RFdiffusion2 (Dauparas 2025) — VERIFY current public release/repo at generation time",
    "Riff-Diff (Schnettler 2025, Nature) — VERIFY current public release/repo at generation time",
    "CLEAN (CLEAN-style EC/functional classification) — VERIFY current release/repo at generation time",
]

for name, url in STABLE_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"[{r.status_code}] {name}\n      {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e!r}\n      {url}")
print("\nVERIFY MANUALLY (do not assert a repo URL you are unsure of):")
for v in VERIFY_UPSTREAMS:
    print("  -", v)

## 1 · Build the metal-site theozyme and scaffold it (large pool)
The mock path returns placeholder backbones so the loop runs anywhere. On an A100, switch `METHOD` to
`"rfdiffusion2"` or `"riffdiff"` (verify the release) and `N_SCAFFOLDS` to 1000s–10k.

> **A100 NOTE:** scaffolding a **large pool** (GRACE used ~10k) is the compute bottleneck — and metal-
> site placement is harder than a sidechain motif, so budget extra backbones (many will not hold a
> clean tetrahedral His₃ cage). Free Colab T4 can do a small **RFdiffusion** (classic) metal-motif
> demo (tens of backbones); the real campaign wants an A100 (Colab Pro+) or HPC. The mock backend
> below needs no GPU at all.

In [ ]:
from enzyme_tools import build_theozyme, scaffold_motif

theo = build_theozyme("co2_hydration")

METHOD = "mock"        # -> "rfdiffusion2" | "riffdiff" | "rfdiffusion" on Colab/HPC (verify release)
N_SCAFFOLDS = 12       # -> 1000s-10k for the real campaign (GRACE used ~10k)

scaffolds = scaffold_motif(theo, n=N_SCAFFOLDS, method=METHOD)
print(f"{len(scaffolds)} scaffolds via method={METHOD!r} (mock numbers are SYNTHETIC); "
      f"metal={scaffolds[0]['metal']}")
print("example:", scaffolds[0])

## 2 · Metal-aware LigandMPNN — FIXING the three His ligands (+ a ProteinMPNN baseline)
This is the core of **metal-aware** sequence design: redesign the protein but **keep the three His
ligands fixed** AND pass the **Zn as atom context** so the pocket around the charged metal is designed
correctly. That is why **LigandMPNN**, not vanilla ProteinMPNN, is used. We also generate a
**metal-blind ProteinMPNN** set on the same backbones (His fixed, no metal context) for the
notebook-04 benchmark. On Colab set `TOOL="ligandmpnn"` (CPU-fast) with
`--ligand_mpnn_use_atom_context 1` + the fixed-positions list.

In [ ]:
from enzyme_tools import ligandmpnn_metal, proteinmpnn_metal_blind

TOOL = "mock"          # -> "ligandmpnn" on Colab (CPU-fast), with the Zn atom context
SEQS_PER_BACKBONE = 4

ligands = theo.catalytic_residue_ids()   # the three His ligands to FIX
all_designs = []
for bb in scaffolds:
    # metal-AWARE (the real design path):
    for s in ligandmpnn_metal(bb, ligands, n=SEQS_PER_BACKBONE, tool=TOOL):
        s.update(scaffold_id=bb["design_id"], scaffold_method=bb["method"],
                 motif_rmsd=bb["motif_rmsd"], design_tool="ligandmpnn")
        all_designs.append(s)
    # metal-BLIND baseline (for the benchmark only):
    for s in proteinmpnn_metal_blind(bb, ligands, n=SEQS_PER_BACKBONE, tool=TOOL):
        s.update(scaffold_id=bb["design_id"], scaffold_method=bb["method"],
                 motif_rmsd=bb["motif_rmsd"], design_tool="proteinmpnn")
        all_designs.append(s)

n_lig = sum(d["design_tool"] == "ligandmpnn" for d in all_designs)
n_pm = sum(d["design_tool"] == "proteinmpnn" for d in all_designs)
print(f"{len(all_designs)} sequences total: {n_lig} metal-aware LigandMPNN + {n_pm} metal-blind "
      f"ProteinMPNN ({len(scaffolds)} backbones x {SEQS_PER_BACKBONE} each); His ligands fixed: {ligands}")

## 3 · Predict + score (mock metal-ligand geometry), write the results CSV
On Colab, predict each sequence with AF2/ESMFold, read the **active-site pLDDT**, **add the Zn from
the His₃ geometry** (AF2 does not place metals), and compute the real `metal_ligand_geometry`. Here
the mock backend fills SYNTHETIC values so the CSV — the input to notebook 03 — is produced anywhere.
We give the metal-blind ProteinMPNN arm a systematically worse metal-geometry tendency so the
benchmark in notebook 04 has the expected shape (this is a SYNTHETIC teaching effect).

In [ ]:
import pandas as pd, hashlib
from enzyme_tools import metal_ligand_geometry, dock_substrate, active_site_md

rows = []
site = theo.metal_site
for d in all_designs:
    # On Colab, `pred_pdb` is the AF2-predicted PDB (with the Zn built in) for this design; the mock
    # backend keys off the (non-existent) per-design path so each design gets a DISTINCT SYNTHETIC value.
    pred_pdb = f"results/pred/{d['design_id']}.pdb"
    geo = metal_ligand_geometry(pred_pdb, site)        # mock -> SYNTHETIC (varies per design)
    mlrmsd = geo["metal_ligand_rmsd"]
    # SYNTHETIC teaching effect: penalise the metal-BLIND arm so LigandMPNN looks better at the cage.
    if d["design_tool"] == "proteinmpnn":
        mlrmsd = round(mlrmsd + 0.25, 3)
    sub = theo.substrate.split(";")[0].strip()
    dock = dock_substrate(pred_pdb, sub)               # mock -> SYNTHETIC
    md_res = active_site_md(pred_pdb, ns=10.0)         # mock -> SYNTHETIC (metal-FF caveat applies)
    # SYNTHETIC stand-ins for AF2 confidence + a solubility score so the plumbing runs:
    h = int(hashlib.sha256(d["design_id"].encode()).hexdigest(), 16)
    plddt = 78 + (h % 20)             # 78-97, SYNTHETIC
    plddt_cat = 80 + ((h >> 7) % 18)  # 80-97, SYNTHETIC
    scrmsd = round(0.8 + ((h >> 11) % 200) / 100.0, 2)   # 0.8-2.8, SYNTHETIC
    solubility = round(-1.5 + ((h >> 17) % 300) / 100.0, 2)  # -1.5..1.5, SYNTHETIC (CamSol-style)
    rows.append(dict(
        design_id=d["design_id"], scaffold_id=d["scaffold_id"],
        scaffold_method=d["scaffold_method"], design_tool=d["design_tool"],
        metal_aware=d["metal_aware"], sequence=d["sequence"],
        plddt=plddt, plddt_catalytic=plddt_cat, scrmsd=scrmsd,
        metal_ligand_rmsd=mlrmsd, mean_zn_n_dist=geo["mean_zn_n_dist"],
        mean_n_zn_n_angle=geo["mean_n_zn_n_angle"], solubility=solubility,
        vina_score=dock["vina_score"], md_rmsd=md_res["md_rmsd"], synthetic=True))

camp = pd.DataFrame(rows)
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape, "(ALL NUMBERS SYNTHETIC — mock backend)")
print(f"metal_ligand_rmsd range: {camp['metal_ligand_rmsd'].min()}-{camp['metal_ligand_rmsd'].max()} A "
      f"(target < 0.5); columns include design_tool for the LigandMPNN-vs-ProteinMPNN benchmark")
camp.head()

## D2 checklist
- [ ] Scaffolding run logged (method, release/commit, N backbones — toward ~10k, seed) — A100 for the real campaign.
- [ ] Metal-aware LigandMPNN sequences with the **three His ligands provably fixed** + the **Zn passed as atom context** (logged).
- [ ] Metal-blind **ProteinMPNN baseline** generated on the same backbones (for the benchmark).
- [ ] `results/campaign.csv` with one row per design (real metrics on Colab; mock here), carrying `design_tool` + `metal_ligand_rmsd` + `solubility`.
- [ ] Design log (every config + seed + output path) + 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared enzyme filter on `campaign.csv`.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — the shared enzyme filter (metal-site geometry)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 20** you filter on the **enzyme** cutoffs, with the **metal-ligand
geometry** (mapped onto `catalytic_geom_rmsd`) as the decisive metric, plus a solubility check and a
GRACE-style **CLEAN-style functional classification**.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
Improvements here are pull-requested back to `shared/` for the whole cohort — do not silently fork it.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("enzyme cutoffs:", fp.DEFAULT_CUTOFFS["enzyme"])
print("(catalytic_geom_rmsd <= 0.5 is filled here by the METAL-LIGAND RMSD)")

## Build `fp.Design` objects (enzyme) from the campaign
Map each campaign row onto an `fp.Design`, carrying the enzyme-specific fields: `plddt`,
**`plddt_catalytic`**, `scrmsd`, **`catalytic_geom_rmsd`** (= the **metal-ligand RMSD**), `solubility`,
and `md_rmsd`. The self-consistency layer checks these against the enzyme cutoffs (scrmsd ≤ 2.0,
plddt ≥ 85, plddt_cat ≥ 90, cat_geom ≤ 0.5); the physics layer checks solubility.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence=str(r.get("sequence", "")),
        design_type="enzyme",
        plddt=float(r["plddt"]),
        plddt_catalytic=float(r["plddt_catalytic"]),
        scrmsd=float(r["scrmsd"]),
        catalytic_geom_rmsd=float(r["metal_ligand_rmsd"]),   # METAL-LIGAND geometry is the key metric
        solubility=float(r["solubility"]),
        md_rmsd=float(r["md_rmsd"]),
        extra={"scaffold_method": r["scaffold_method"], "design_tool": r["design_tool"],
               "metal_aware": bool(r["metal_aware"]), "synthetic": True},
    ))
print(len(designs), "enzyme Design objects built (from SYNTHETIC mock metrics)")

## Run the pipeline (`design_type="enzyme"`) and report
`run_pipeline` applies the layers in order and returns a ranked DataFrame. We use layers 1+3+4
(self-consistency incl. metal-ligand geometry, physics incl. solubility, and the caveated short-MD
dynamics layer); the orthogonal layer (L2) needs a second predictor's scRMSD, which you add on Colab.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="enzyme", use_layers=(1, 3, 4))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj20")
top

## GRACE-style functional triage: a CLEAN-style classification (mock)
GRACE pairs geometry with a **functional classifier** (CLEAN-style EC/function prediction) + solubility
to prune a large pool. The real call runs CLEAN on each sequence; here a deterministic mock flags each
design as the intended class or not, so the plumbing runs. **SYNTHETIC** — never report as real.

In [ ]:
import hashlib

def clean_style_classification_mock(seq, design_id):
    """SYNTHETIC stand-in for a CLEAN-style EC/functional classifier (verify + wire up the real tool).
    Returns (predicted_class, confidence). On Colab: run CLEAN and parse its EC prediction."""
    h = int(hashlib.sha256(("clean" + str(design_id)).encode()).hexdigest(), 16)
    is_ca = (h % 100) < 65          # SYNTHETIC: ~65% read as the intended class
    conf = round(0.5 + (h % 50) / 100.0, 2)
    return ("carbonic-anhydrase-like (EC 4.2.1.1)" if is_ca else "other/uncertain"), conf

camp["clean_class"], camp["clean_conf"] = zip(*[
    clean_style_classification_mock(s, i) for s, i in zip(camp["sequence"], camp["design_id"])])
camp.to_csv("results/campaign.csv", index=False)
n_ca = (camp["clean_class"].str.startswith("carbonic")).sum()
print(f"CLEAN-style classification [SYNTHETIC]: {n_ca}/{len(camp)} read as carbonic-anhydrase-like")
print("On Colab: replace with the real CLEAN EC prediction; pair it with solubility to prune the pool.")

## Survival-at-each-layer + metal-geometry pass rate (honest accounting)
The **metal-geometry layer is where most metalloenzyme designs die** — expect the steepest drop there.
Report the pass rate explicitly; this is a headline benchmark for D3.

In [ ]:
print("layers_passed distribution:")
print(df_ranked["layers_passed"].value_counts().sort_index())

n = len(df_ranked)
cut = fp.DEFAULT_CUTOFFS["enzyme"]["cat_geom"]
n_geom = int((df_ranked["catalytic_geom_rmsd"] <= cut).sum())
print(f"\nmetal-geometry preservation: {n_geom}/{n} "
      f"({100*n_geom/max(n,1):.1f}%) hold the Zn-His3 cage < {cut} A  [SYNTHETIC demo numbers]")
print("Reminder: geometry != metal incorporation != activity. ICP + an assay decide (notebook 05).")

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module with `design_type="enzyme"`.
- [ ] Survival-at-each-layer figure (`results/proj20_survival.png`).
- [ ] Metal-geometry preservation rate reported (the headline metric).
- [ ] Solubility + CLEAN-style functional classification added (GRACE-style triage).
- [ ] Mapping assumptions written down (metal-ligand RMSD → `catalytic_geom_rmsd`; AF2 doesn't place the Zn).

**Next:** `04_validate.ipynb` — metal-geometry + LigandMPNN-vs-ProteinMPNN + pool-size-vs-hit-rate + docking/MD.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — metal geometry, LigandMPNN-vs-ProteinMPNN, pool-size-vs-hit-rate, docking & MD

**Standard slot:** *validate (in silico).* **For Project 20 these are the benchmarks:** the
**metal-geometry preservation rate**, **LigandMPNN vs ProteinMPNN at the metal site**, and
**pool-size vs hit-rate** `[extension]`, plus the docking / caveated metal-site-MD figures (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Metal-geometry preservation — the headline figure
Distribution of metal-ligand RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate** — the metric that most distinguishes design tools and scaffolding methods.
(Numbers here are SYNTHETIC mock values; on Colab they come from real AF2 predictions with the Zn
built in.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["metal_ligand_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("metal-ligand RMSD vs target Zn-His3 (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Metal-geometry preservation")
ax.legend(); plt.tight_layout()
plt.savefig("results/metal_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["metal_ligand_rmsd"] <= cut).mean()
print(f"overall metal-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")

## 2 · LigandMPNN vs ProteinMPNN at the metal site — the core benchmark
Does the **metal context** (LigandMPNN) hold the Zn-His₃ cage better than a **metal-blind** design
(ProteinMPNN, His fixed but no metal in context)? Compare the metal-geometry preservation rate by
`design_tool`. (Here the mock data gives LigandMPNN the edge by construction — a SYNTHETIC teaching
effect; on Colab the difference is whatever the real designs show.)

In [ ]:
by_tool = (camp.assign(pass_geom=camp["metal_ligand_rmsd"] <= 0.5)
               .groupby("design_tool")
               .agg(n=("design_id", "size"),
                    geom_pass_rate=("pass_geom", "mean"),
                    mean_metal_rmsd=("metal_ligand_rmsd", "mean"),
                    mean_plddt_cat=("plddt_catalytic", "mean"))
               .reset_index())
by_tool["geom_pass_rate"] = (100 * by_tool["geom_pass_rate"]).round(1)
print("LigandMPNN (metal-aware) vs ProteinMPNN (metal-blind) at the Zn-His3 site [SYNTHETIC]:")
print(by_tool.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.bar(by_tool["design_tool"], by_tool["geom_pass_rate"])
ax.set_ylabel("metal-geometry pass rate (%)  [SYNTHETIC]")
ax.set_title("Metal context helps hold the cage")
for i, v in enumerate(by_tool["geom_pass_rate"]):
    ax.text(i, v, f"{v}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig("results/ligandmpnn_vs_proteinmpnn.png", dpi=150); plt.show()
print("On Colab: run BOTH tools on the SAME backbones (His fixed); the only difference is the Zn context.")

## 3 · Pool-size vs hit-rate — does scaling toward ~10k pay off? `[extension]`
GRACE generated a **large pool (~10k)**. Subsample the pool at increasing sizes and plot the cumulative
count of designs passing the metal-geometry bar. A roughly linear count (flat *rate*) means hits scale
with pool size — the rationale for a large campaign. (SYNTHETIC mock pool here; the *shape* is the point.)

In [ ]:
rng = np.random.default_rng(0)
lig = camp[camp["design_tool"] == "ligandmpnn"].copy()
passes = (lig["metal_ligand_rmsd"] <= 0.5).to_numpy()
order = rng.permutation(len(passes))
sizes = np.linspace(1, len(passes), min(20, len(passes))).astype(int)
cum_hits = [int(passes[order[:k]].sum()) for k in sizes]

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(sizes, cum_hits, marker="o")
ax.set_xlabel("pool size (designs screened)  [SYNTHETIC mock pool]")
ax.set_ylabel("cumulative metal-geometry hits")
ax.set_title("Pool-size vs hit count (extrapolate toward ~10k)")
plt.tight_layout(); plt.savefig("results/pool_size_vs_hits.png", dpi=150); plt.show()
rate = 100 * passes.mean()
print(f"hit RATE is ~constant at {rate:.1f}% [SYNTHETIC] -> hit COUNT grows with pool size; "
      "this is why GRACE used ~10k. On Colab, plot your real pool.")

## 4 · Substrate fit + metal-site MD (top candidates) — with the metal-FF caveat
Docking (AutoDock Vina) checks the substrate (CO₂ / the pNPA proxy) **reaches and is oriented toward
the Zn-OH** — not affinity, not activity. Short MD (OpenMM) checks the metal site doesn't drift — **but
classical fixed-charge force fields model a coordinated metal poorly**, so this is a *weak proxy*, not
ground truth. Plot the two for the ranked survivors as orthogonal (caveated) evidence.

In [ ]:
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(camp[["design_id", "vina_score"]], on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.2, 3.4))
sc = ax.scatter(top["vina_score"], top["md_rmsd"],
                c=top["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("Vina substrate-fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("metal-site MD RMSD (A)  [SYNTHETIC; metal-FF caveat]")
ax.set_title("Top candidates: substrate fit vs metal-site stability")
fig.colorbar(sc, label="metal-ligand RMSD (A)")
plt.tight_layout(); plt.savefig("results/docking_md.png", dpi=150); plt.show()
print("Lower-left + dark points (good fit, stable, good cage) are best [SYNTHETIC].")
print("CAVEAT: classical MD cannot model the Zn centre well — treat 'stable' as necessary, not sufficient.")

## 5 · Honest hit-rate accounting
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate is **not** an activity rate, **and not even a metal-incorporation rate**. Geometry ≠
incorporation ≠ catalysis; ICP and a kinetic assay are required.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["metal_ligand_rmsd"] <= 0.5).sum())
print("Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated             : {n_total}")
print(f"  pass all filter layers: {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo metalloenzyme activity rates are low (GRACE needed ~10k + screening).")
print("In-silico metal geometry does NOT guarantee activity, NOR that Zn binds. ICP + an assay decide.")

## D3 (part 2) checklist
- [ ] Metal-geometry preservation histogram (`results/metal_geometry_hist.png`) + rate.
- [ ] **LigandMPNN-vs-ProteinMPNN** metal-site benchmark (`results/ligandmpnn_vs_proteinmpnn.png`).
- [ ] **Pool-size-vs-hit-rate** figure (`results/pool_size_vs_hits.png`) `[extension]`.
- [ ] Docking + (caveated) metal-site-MD figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ incorporation ≠ activity" + metal-FF caveats stated.

**Next:** `05_validation_plan.ipynb` — the activity + metal-incorporation assay plan + controls + Co-substitution stretch.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation plan — activity + metal-incorporation assay, controls, Co substitution

**Standard slot:** *validation plan.* **For Project 20 this means:** turn the metal-geometry-filtered
set into a costed **activity + metal-incorporation** assay plan with the right controls (incl. an
**apo** enzyme and a **natural CA**), and an **alternative-metal Co(II) substitution** test for hits
(D4/D5).

This is the deliverable that states, plainly: **in-silico geometry is a hypothesis; the assay tests
activity and ICP tests whether the metal even binds.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the ranked filter (metal-geometry first), capped at **< 96** so they fit a
single screening plate with controls. Diversity matters — don't pick 96 near-identical designs.

In [ ]:
import pandas as pd
try:
    ranked = pd.read_csv("results/ranked.csv")
except FileNotFoundError:
    ranked = pd.read_csv("results/campaign.csv")

# Prefer designs passing the metal-geometry bar; cap under 96 (leave wells for controls).
ok = ranked[ranked["catalytic_geom_rmsd"] <= 0.5] if "catalytic_geom_rmsd" in ranked else ranked
selected = ok.head(84).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds; record why each was chosen in your report.")

## 2 · Metal incorporation FIRST — it gates everything
A design with no bound Zn cannot be a metalloenzyme, regardless of geometry. Check incorporation
**before** trusting any activity:
- **ICP-MS** — bound metal (Zn) per protein; the quantitative gold standard.
- **PAR / 4-(2-pyridylazo)resorcinol** colorimetric assay — release the metal with a chelator and read
  it colorimetrically; a quick bench check of Zn stoichiometry.

Express in *E. coli* BL21(DE3), 16–18 °C overnight; His-tag → IMAC → SEC; **supplement Zn(II)** in the
medium/buffer to favour loading. Report **Zn : protein stoichiometry** — aim near 1:1.

## 3 · The activity assays (esterase proxy + true CO₂ hydration)
- **Esterase proxy (fast bench readout):** **p-nitrophenyl acetate (pNPA)** hydrolysis → follow
  **p-nitrophenolate** absorbance (~348–405 nm) in a plate reader; initial rates across substrate
  concentrations. Convenient chromogenic kinetics (CA has a promiscuous esterase activity).
- **True CO₂ hydration:** the classic **Wilbur-Anderson** assay — time the **pH drop** as CO₂ is
  hydrated; report **WA units** (stopped-flow for fast designs).
- **Note:** the esterase proxy and CO₂-hydration activity are **correlated but not identical** — state
  which you measured. Subtract the uncatalysed background (both reactions proceed without enzyme).

In [ ]:
controls = {
    "POSITIVE — natural carbonic anhydrase": "a verified CA (e.g. bovine/human CA II); confirms the assay works",
    "NEGATIVE — apo enzyme (metal stripped)": "SAME design, Zn removed with a chelator; the metalloenzyme "
        "analogue of a dead mutant — loss of activity pins catalysis to the metal",
    "NEGATIVE — His -> Ala metal-knockout mutant": "SAME design, a His ligand mutated to Ala; cannot bind the metal",
    "NEGATIVE — empty-vector lysate": "no insert; rules out host-background activity",
    "BLANK — buffer + substrate only": "the uncatalysed CO2-hydration / pNPA background rate to subtract",
}
print("MANDATORY controls (every plate):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")
print("\nMETAL-INCORPORATION gate: run ICP-MS / PAR on every candidate BEFORE trusting its activity.")

## 4 · A costed, plate-based screen (template — fill real prices)
One 96-well plate holds the < 96 designs + the controls above. Cost the gene synthesis, expression,
purification, the **ICP metal check**, and assay reagents at your institution's rates.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis (Zn-supplemented)", "1 plate", "fill"),
    ("IMAC + SEC purification (plate format)", "1 plate", "fill"),
    ("ICP-MS / PAR metal-incorporation check", "per candidate", "fill"),
    ("p-nitrophenyl acetate (pNPA) substrate", "stock", "fill"),
    ("Wilbur-Anderson CO2 assay (gas + pH stack)", "per design", "fill"),
    ("Plate-reader / stopped-flow time (kinetics)", "per plate", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:52s} {scale:14s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+ICP+assay 2-3 wk.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here, "
      "but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 5 · Alternative metal — Co(II) substitution `[stretch]`
Zn(II) is **spectroscopically silent**; **Co(II)** is active in carbonic anhydrase **and** gives a
diagnostic UV-vis d-d band. So a Co-substituted version is a powerful orthogonal confirmation that the
designed site is a **genuine metal site**:
- **Reconstitute:** strip to apo (chelator), then add Co(II); confirm uptake by ICP.
- **Spectroscopy:** look for the Co(II) d-d absorption band (a CA-like signature).
- **Activity:** measure pNPA / CO₂ activity with Co(II) vs Zn(II) vs apo.
A Co-restored activity + the expected band, with the apo form dead, is strong evidence the site is
real — much more than geometry alone.

In [ ]:
print("Co(II)-substitution loop (stretch): apo (strip Zn) -> add Co(II) -> ICP confirms uptake ->")
print("UV-vis for the Co(II) d-d band -> pNPA/CO2 activity (Co vs Zn vs apo).")
print("Reminder for the thesis: report Zn:protein stoichiometry, the hit rate, and that geometry,")
print("metal incorporation, and activity are THREE separate claims — each needs its own evidence.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, metal-geometry-passing designs.
- [ ] **Metal-incorporation** plan (ICP-MS / PAR) run BEFORE activity — report Zn:protein stoichiometry.
- [ ] Activity-assay plan: pNPA esterase proxy and/or Wilbur-Anderson CO₂ units, with **all** controls
      (apo, His→Ala knockout, natural CA, empty vector, blank), costed + timed.
- [ ] Alternative-metal **Co(II) substitution** test for hits `[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "geometry ≠ incorporation ≠ activity" + the metal-FF caveat stated plainly.

You're done — this is the **enzyme-family template** (Project 18) specialised to a **metal active site**.